# Data collection for the year 2022

In [1]:
import cocopp
dsl = cocopp.load("bbob/2022/*")

  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2022/CMA-ES-Akimoto_Gharafi.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMA-ES-Akimoto_Gharafi.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2022/CMA-ES-pycma_Gharafi.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMA-ES-pycma_Gharafi.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2022/CMAES-APOP-KMA_Nguyen.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KMA_Nguyen.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2022/CMAES-APOP-KP_Nguyen.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KP_Nguyen.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2022/CMAES-APOP-MA_Nguyen.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-MA_Nguyen.tgz
  downloading https://nu

In [2]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> oMads-Neg_Dahito  (ERT=62.1 @ 1e-08)
dim= 2, F 2 -> oMads-Neg_Dahito  (ERT=276 @ 1e-08)
dim= 2, F 3 -> oMads-Neg_Dahito  (ERT=643 @ 1e-08)
dim= 2, F 4 -> oMads-Neg_Dahito  (ERT=1.41e+03 @ 1e-08)
dim= 2, F 5 -> oMads-2N_Dahito  (ERT=6.27 @ 1e-08)
dim= 2, F 6 -> oMads-Neg_Dahito  (ERT=219 @ 1e-08)
dim= 2, F 7 -> oMads-Neg_Dahito  (ERT=279 @ 1e-08)
dim= 2, F 8 -> oMads-Neg_Dahito  (ERT=212 @ 1e-08)
dim= 2, F 9 -> oMads-Neg_Dahito  (ERT=184 @ 1e-08)
dim= 2, F10 -> DD-CMA-ES-pycma_Gharafi  (ERT=474 @ 1e-08)
dim= 2, F11 -> DD-CMA-ES-pycma_Gharafi  (ERT=460 @ 1e-08)
dim= 2, F12 -> oMads-Neg_Dahito  (ERT=443 @ 1e-08)
dim= 2, F13 -> DD-CMA-ES-pycma_Gharafi  (ERT=586 @ 1e-08)
dim= 2, F14 -> CMA-ES-pycma_Gharafi  (ERT=456 @ 1e-08)
dim= 2, F15 -> oMads-Neg_Dahito  (ERT=1.68e+03 @ 1e-08)
dim= 2, F16 -> oMads-Neg_Dahito  (ERT=566 @ 1e-08)
dim= 2, F17 -> DD-CMA-ES-pycma_Gharafi  (ERT=1.7e+03 @ 1e-08)
dim= 2, F18 -> CMA-ES-pycma_Gharafi  (ERT=2.41e+03 @ 1e-08)
dim= 2, F19 -> oMads-Neg_D

In [3]:
from collections import Counter, defaultdict

In [4]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

oMads-Neg_Dahito: 25
CMAES-APOP-KMA_Nguyen: 22
CMAES-APOP-MA_Nguyen: 21
oMads-2N_Dahito: 18
DD-CMA-ES-pycma_Gharafi: 17
CMA-ES-pycma_Gharafi: 13
CMA-ES-Akimoto_Gharafi: 11
DD-CMA-ES-Akimoto_Gharafi: 9
CMAES-APOP-KP_Nguyen: 3


In [5]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'oMads-Neg_Dahito',
 3: 'oMads-Neg_Dahito',
 5: 'CMA-ES-pycma_Gharafi',
 10: 'CMAES-APOP-KMA_Nguyen',
 20: 'CMAES-APOP-KMA_Nguyen',
 40: 'CMAES-APOP-MA_Nguyen'}

In [6]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20 40]
    dimension  function_id        target    best_algorithm    best_ERT
0           2            1  1.000000e-08  oMads-Neg_Dahito   62.133333
1           2            1  1.000000e-05  oMads-Neg_Dahito   53.066667
2           2            1  1.000000e-03   oMads-2N_Dahito   23.933333
3           2            1  1.000000e-02   oMads-2N_Dahito   23.933333
4           2            1  1.000000e-01   oMads-2N_Dahito   16.733333
5           2            2  1.000000e-08  oMads-Neg_Dahito  276.133333
6           2            2  1.000000e-05  oMads-Neg_Dahito  178.066667
7           2            2  1.000000e-03   oMads-2N_Dahito  128.333333
8           2            2  1.000000e-02   oMads-2N_Dahito  110.266667
9           2            2  1.000000e-01   oMads-2N_Dahito   85.533333
10          2            3  1.000000e-08  oMads-Neg_Dahito  643.400000
11          2            3  1.000000e-05  oMads-Neg_Dahito  587.600000
12          2            3 

In [7]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2022.csv", index=False)
